In [3]:
import pandas as pd
df = pd.read_csv('masked_plant_pest.csv')
df.head(30)

,id,observed_on,latitude,longitude,phenophase,scientific_name,common_name,taxon_id,doy,biome,T2M,PRECTOTCORR,SWGDN,biome_cat,scientific_name_pest,common_name_pest
0,310657266,8/31/2025,40.799434,-111.012697,Flowering,Achillea millefolium,common yarrow,52821,243,26,292.79025,0.000011,288.58110,Dfd,Empoasca fabae,Potato Leafhopper
1,310812803,8/31/2025,40.690640,-110.903167,Flowering,Achillea millefolium,common yarrow,52821,243,27,290.41034,0.000012,279.16245,ET,Ostrinia nubilalis,European Corn Borer Moth
2,310648485,8/31/2025,43.939639,-87.719908,Flowering,Achillea millefolium,common yarrow,52821,243,26,294.50253,0.000030,255.68756,Dfd,Popillia japonica,Japanese Beetle
3,310647240,8/31/2025,44.741903,-65.519220,Flowering,Achillea millefolium,common yarrow,52821,243,26,290.05582,0.000014,265.07320,Dfd,Lymantria dispar,Spongy Moth
4,310482483,8/31/2025,42.146311,-77.131258,Flowering,Achillea millefolium,common yarrow,52821,243,25,293.99230,0.000021,266.84604,Dfc,Leptinotarsa decemlineata,Colorado Potato Beetle
5,310457618,8/31/2025,58.544962,31.377577,Flowering,Achillea millefolium,common yarrow,52821,243,26,288.90690,0.000045,192.62874,Dfd,Leptinotarsa decemlineata,Colorado Potato Beetle
6,310215347,8/30/2025,63.558598,26.662874,Flowering,Achillea millefolium,common yarrow,52821,242,27,287.83690,0.000021,182.90506,ET,Plutella xylostella,Diamondback Moth
7,310201801,8/30/2025,60.234195,24.946592,Flowering,Achillea millefolium,common yarrow,52821,242,26,289.69420,0.000028,196.11778,Dfd,Plutella xylostella,Diamondback Moth
8,311993972,8/30/2025,42.558192,2.112421,Flowering,Achillea millefolium,common yarrow,52821,242,26,290.25610,0.000025,251.79330,Dfd,Lymantria dispar,Spongy Moth
9,311993941,8/30/2025,42.419870,2.010164,Flowering,Achillea millefolium,common yarrow,52821,242,26,290.25610,0.000025,251.79330,Dfd,Lymantria dispar,Spongy Moth


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib
import json

# ---------------- 1. Load Data ----------------
file_name = 'masked_plant_pest.csv'

try:
    df = pd.read_csv(file_name, low_memory=False, dtype=str)
    print(f"Loaded {len(df)} rows.")
except FileNotFoundError:
    print(f"Error: '{file_name}' not found.")
    exit()

# ---------------- 2. Clean & Preprocess ----------------
# Clean phenophase
df['phenophase'] = df['phenophase'].astype(str).str.strip().str.replace(r'[\[\]\'"]', '', regex=True)

# Ensure numeric columns
for col in ['T2M', 'PRECTOTCORR', 'SWGDN', 'doy']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert date
df['observed_on'] = pd.to_datetime(df['observed_on'], errors='coerce')

# Drop rows with missing crucial info
df.dropna(subset=['phenophase', 'scientific_name', 'biome_cat', 'doy'], inplace=True)

# Day of Year
df['DOY'] = df['observed_on'].dt.dayofyear
Target: Lateness ---
# ---------------- 3. -------------
mean_doy = df.groupby(['scientific_name', 'phenophase'])['DOY'].mean().reset_index()
mean_doy.rename(columns={'DOY': 'Expected_DOY'}, inplace=True)
df = pd.merge(df, mean_doy, on=['scientific_name', 'phenophase'], how='left')
df['Lateness'] = df['DOY'] - df['Expected_DOY']

# ---------------- 4. Categorical Encoding ----------------
for col in ['scientific_name', 'phenophase', 'biome_cat']:
    df[col] = df[col].astype('category')

df['Species_Encoded'] = df['scientific_name'].cat.codes
df['Phenophase_Encoded'] = df['phenophase'].cat.codes
df['Biome_Encoded'] = df['biome_cat'].cat.codes

species_cats = df['scientific_name'].cat.categories.tolist()
phenophase_cats = df['phenophase'].cat.categories.tolist()
biome_cats = df['biome_cat'].cat.categories.tolist()

# ---------------- 5. Features & Target ----------------
features = ['Species_Encoded', 'Phenophase_Encoded', 'Biome_Encoded', 'T2M', 'PRECTOTCORR', 'SWGDN']
X = df[features]
y = df['Lateness']

# Drop any remaining NaNs
X.dropna(inplace=True)
y = y[X.index]

# ---------------- 6. Train Model ----------------
model = xgb.XGBRegressor(
    n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1,
    objective='reg:squarederror', tree_method='gpu_hist'
)
model.fit(X, y)

# ---------------- 7. Save Model & Encoders ----------------
joblib.dump(model, 'plant_phenology_model.joblib')
joblib.dump(species_cats, 'species_cats.joblib')
joblib.dump(phenophase_cats, 'phenophase_cats.joblib')
joblib.dump(biome_cats, 'biome_cats.joblib')

json.dump({
    'species': species_cats,
    'phenophases': phenophase_cats,
    'biomes': biome_cats
}, open('plant_categories.json', 'w'), indent=4)

# ---------------- 8. Batch Prediction Function ----------------
# ---------------- 8. Batch Prediction Function ----------------
def predict_days_batch(species_list, phenophase_list, biome_list, T2M_list, PRECTOTCORR_list, SWGDN_list):
    results = []
    
    def encode(value, categories):
        if value in categories:
            return categories.index(value)
        else:
            raise ValueError(f"Value '{value}' not found in categories.")
    
    for sp, ph, bm, t2m, pr, sw in zip(species_list, phenophase_list, biome_list, T2M_list, PRECTOTCORR_list, SWGDN_list):
        try:
            s_code = encode(sp, species_cats)
            p_code = encode(ph, phenophase_cats)
            b_code = encode(bm, biome_cats)
        except ValueError as e:
            results.append(str(e))
            continue
        
        x_input = pd.DataFrame([[s_code, p_code, b_code, t2m, pr, sw]], columns=features)
        pred = model.predict(x_input)[0]
        pred_days = int(round(pred))  # rounded integer
        
        if pred_days > 0:
            results.append(f"{pred_days} days late")
        elif pred_days < 0:
            results.append(f"{abs(pred_days)} days early")
        else:
            results.append("On time")
    
    return results


Loaded 653789 rows.


C:\Users\aayus\AppData\Local\Temp\ipykernel_8036\3950353577.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.dropna(inplace=True)
d:\Programing\Machine learning\venv\Lib\site-packages\xgboost\core.py:160: UserWarning: [03:25:54] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0b3782d1791676daf-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
d:\Programing\Machine learning\venv\Lib\site-packages\xgboost\core.py:160: UserWarning: [03:25:55] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0b3782d1791676daf-1\xgboost\xgboost-ci-windows\

In [19]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib
import json

# --- 1. Load Data ---
file_name = 'masked_plant_pest.csv' 
df = pd.read_csv(file_name, low_memory=False, dtype=str)

# Clean essential columns
df['common_name'] = df['common_name'].astype(str).str.strip()
df['scientific_name'] = df['scientific_name'].astype(str).str.strip()
df['phenophase_clean'] = df['phenophase'].astype(str).str.strip().str.replace(r'[\[\]\'"]', '', regex=True)
df['phenophase'] = df['phenophase_clean']

df['observed_on'] = pd.to_datetime(df['observed_on'], errors='coerce')
df.dropna(subset=['observed_on','scientific_name','phenophase'], inplace=True)
df['DOY'] = df['observed_on'].dt.dayofyear

# --- 2. Target Variable ---
mean_doy = df.groupby(['scientific_name','phenophase'])['DOY'].mean().reset_index()
mean_doy.rename(columns={'DOY':'Expected_DOY'}, inplace=True)
df = pd.merge(df, mean_doy, on=['scientific_name','phenophase'], how='left')
df['Lateness'] = df['DOY'] - df['Expected_DOY']

# --- 3. Feature Encoding ---
df['scientific_name'] = df['scientific_name'].astype('category')
df['phenophase'] = df['phenophase'].astype('category')
df['biome_cat'] = df['biome_cat'].astype('category')

df['Species_Encoded'] = df['scientific_name'].cat.codes
df['Phenophase_Encoded'] = df['phenophase'].cat.codes
df['Biome_Encoded'] = df['biome_cat'].cat.codes

species_cats = list(df['scientific_name'].cat.categories)
phenophase_cats = list(df['phenophase'].cat.categories)
biome_cats = list(df['biome_cat'].cat.categories)

# --- 4. Features & Target ---
features = ['Species_Encoded','Phenophase_Encoded','Biome_Encoded','T2M','PRECTOTCORR','SWGDN']
df['T2M'] = pd.to_numeric(df['T2M'], errors='coerce')
df['PRECTOTCORR'] = pd.to_numeric(df['PRECTOTCORR'], errors='coerce')
df['SWGDN'] = pd.to_numeric(df['SWGDN'], errors='coerce')
df.dropna(subset=features, inplace=True)

X = df[features]
y = df['Lateness']

# --- 5. Train-Test Split & Model ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    objective='reg:squarederror',
    tree_method='gpu_hist',
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.2f} days")

# Save model and encoders
joblib.dump(model,'plant_model.joblib')
joblib.dump(species_cats,'species_cats.joblib')
joblib.dump(phenophase_cats,'phenophase_cats.joblib')
joblib.dump(biome_cats,'biome_cats.joblib')

# --- 6. Batch Prediction Function ---
def predict_days_batch(species_list, phenophase_list, biome_list, T2M_list, PRECTOTCORR_list, SWGDN_list):
    preds = []
    for sp, ph, bm, t2m, pr, sw in zip(species_list, phenophase_list, biome_list, T2M_list, PRECTOTCORR_list, SWGDN_list):
        # Encoding
        try:
            s_code = species_cats.index(sp)
            p_code = phenophase_cats.index(ph)
            b_code = biome_cats.index(bm)
        except ValueError:
            preds.append("Unknown species/phenophase/biome")
            continue
        # Prepare row
        row = pd.DataFrame({
            'Species_Encoded':[s_code],
            'Phenophase_Encoded':[p_code],
            'Biome_Encoded':[b_code],
            'T2M':[t2m],
            'PRECTOTCORR':[pr],
            'SWGDN':[sw]
        })
        lateness = model.predict(row)[0]
        lateness_days = round(abs(lateness))
        if lateness >= 0:
            preds.append(f"{lateness_days} days late")
        else:
            preds.append(f"{lateness_days} days early")
    return preds



d:\Programing\Machine learning\venv\Lib\site-packages\xgboost\core.py:160: UserWarning: [03:26:01] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0b3782d1791676daf-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


MAE: 33.73 days


d:\Programing\Machine learning\venv\Lib\site-packages\xgboost\core.py:160: UserWarning: [03:26:01] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0b3782d1791676daf-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


In [22]:
# 10 species ke naam
species_list = [
    "Achillea millefolium", "Acer negundo", "Acer platanoides",
    "Acer pseudoplatanus", "Acer rubrum", "Actinotus helianthi",
    "Aegopodium podagraria", "Aesculus hippocastanum",
    "Ageratina altissima", "Agrimonia eupatoria"
]

# Phenophase for each species
phenophase_list = [
    "Flowering", "Flowering", "Fruiting", "Flowering", "Flowering",
    "Flowering", "Flowering", "Flowering", "Flowering", "Flowering"
]

# Constant environmental values for all species
biome_list = ["Dfd"]*10
T2M_list = [292.5]*10          # Same temperature for all
PRECTOTCORR_list = [1.2e-5]*10 # Same precipitation for all
SWGDN_list = [280.0]*10        # Same soil water/growth data for all

# Call batch prediction
pred_days = predict_days_batch(
    species_list, phenophase_list, biome_list,
    T2M_list, PRECTOTCORR_list, SWGDN_list
)

# Print just the days
for days in pred_days:
    print(f"{days} days")



1 days early days
2 days late days
15 days early days
2 days late days
21 days late days
1 days early days
1 days early days
1 days early days
12 days early days
12 days early days


In [21]:
# --- 7. Accuracy Check on Test Data ---
def check_model_accuracy(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_pred_days = np.round(y_pred)  # Rounded days
    y_true_days = np.round(y_test.values)
    
    # Mean Absolute Error
    mae = np.mean(np.abs(y_pred_days - y_true_days))
    print(f"Mean Absolute Error (rounded days): {mae:.2f} days")
    
    # Optional: Show sample of predictions vs actual
    comparison = pd.DataFrame({
        'Predicted (days)': y_pred_days,
        'Actual (days)': y_true_days,
        'Difference': y_pred_days - y_true_days
    })
    print("\nSample predictions vs actual:")
    print(comparison.head(10))  # First 10 rows for example
    
    return mae, comparison

# Call accuracy check
mae_value, comparison_df = check_model_accuracy(model, X_test, y_test)


Mean Absolute Error (rounded days): 33.72 days

Sample predictions vs actual:
   Predicted (days)  Actual (days)  Difference
0             -16.0           -2.0       -14.0
1             -16.0           19.0       -35.0
2              37.0           51.0       -14.0
3              18.0           58.0       -40.0
4            -161.0         -153.0        -8.0
5               1.0           24.0       -23.0
6             -23.0          -73.0        50.0
7             -10.0           10.0       -20.0
8               1.0           95.0       -94.0
9               5.0           -1.0         6.0
